# `rudof generate`: one generator, three schema languages

This notebook walks through the synthetic data generator of [rudof](https://github.com/rudof-project/rudof)
from Python (`pyrudof`):

1. generate RDF from a ShEx schema;
2. generate **the same data** from equivalent ShEx, SHACL and PG-Schema schemas;
3. write the same graph **both as RDF and as a property graph**;
4. generate from a published schema (LDBC SNB) and measure its structuredness;
5. break the graph on purpose and read the **report** of what a correct validator must flag,
   in each language;
6. see how the languages' defaults change that report.

The generator reads every schema into one intermediate representation, grounded in the
common graph model of SHACL, ShEx and PG-Schema (Ahmetaj et al., WWW 2025), and generates
from it, so the output does not depend on the language the schema was written in.

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from tempfile import mkdtemp

from pyrudof import (
    DataGenerator, GeneratorConfig, OutputFormat, PgSchemaFormat, RDFFormat, ResultDataFormat, Rudof,
    SchemaFormat, ShaclFormat, ShaclValidationMode, ShapeMapFormat, ShExFormat,
)

# The example schemas shipped with rudof_generate (run this notebook from a rudof checkout).
EXAMPLES = Path.cwd().resolve()
while not (EXAMPLES / "rudof_generate").exists() and EXAMPLES != EXAMPLES.parent:
    EXAMPLES = EXAMPLES.parent
SCHEMAS = EXAMPLES / "rudof_generate" / "examples" / "schemas"
OUT = Path(mkdtemp(prefix="rudof-generate-demo-"))
print(SCHEMAS)
print(sorted(p.name for p in (SCHEMAS / "equivalence").iterdir()))

C:\Users\diego\Documents\paper-knowledge-graph-synthetic-data-generator\rudof\rudof_generate\examples\schemas
['facets.pgs', 'facets.shex', 'facets.ttl', 'keys.pgs', 'keys.shex', 'keys.ttl', 'labels.pgs', 'labels.shex', 'labels.ttl', 'ldbc-snb.pgs', 'ldbc-snb.shex', 'ldbc-snb.ttl', 'references.pgs', 'references.shex', 'references.ttl']


In [2]:
def config(entities=30, seed=7, **output):
    """A generator configuration; `output` sets the output path and format."""
    c = GeneratorConfig()
    c.set_entity_count(entities)
    c.set_seed(seed)
    c.set_write_stats(False)
    c.set_output_path(output.get("path", OUT / "out.ttl"))
    c.set_output_format(output.get("format", OutputFormat.Turtle))
    if "shapemap" in output:
        c.set_shapemap_path(output["shapemap"])
    return c


def ntriples(path):
    """The triples of an RDF file, as a set of N-Triples lines."""
    with Rudof() as rudof:
        rudof.read_data(Path(path).read_text(encoding="utf-8"), RDFFormat.Turtle)
        return {l for l in rudof.serialize_data(ResultDataFormat.NTriples).splitlines() if l.strip()}

## 1. RDF from a ShEx schema

A recursive `:User` shape: every user has a name, may know other users, and has a status
from a value set.

In [3]:
schema = SCHEMAS / "basic" / "person.shex"
print(schema.read_text())
DataGenerator(config(entities=5, path=OUT / "users.ttl")).run_with_format(schema, SchemaFormat.ShEx)
print((OUT / "users.ttl").read_text()[:900])

prefix : <http://example.org/>
prefix xsd: <http://www.w3.org/2001/XMLSchema#>
prefix schema: <http://schema.org/>

start = @:User

:User {
  schema:name   xsd:string             ;
  schema:email  xsd:string           ? ;
  :age          xsd:integer          ? ;
  :knows        @:User               * ;
  :status       [ :Active :Waiting ] ? ;
}

:Person {
  schema:name     xsd:string           ;
  schema:email    xsd:string         ? ;
  :birthDate      xsd:date           ? ;
  :enrolledIn     @:Course           * ;
  :address        @:Address          ? ;
}

:Course {
  schema:name   xsd:string             ;
  :courseCode   xsd:string           ? ;
  :credits      xsd:integer          ? ;
  :instructor   @:Person             ? ;
}

:Address {
  :street       xsd:string             ;
  :city         xsd:string             ;
  :postalCode   xsd:string           ? ;
  :country      xsd:string             ;
}

<http://example.org/Course-1> a <http://example.org/Course> ;
	<http://schema.o

## 2. The same data from three languages

`equivalence/references` describes the running example of the paper — a person works for
an organisation, which is located in a city — in ShEx, SHACL and PG-Schema. The RDF
versions are written over the RDF encoding of the property graph, in which keys, labels
and edges are IRIs under `http://rudof.example/pg/`.

In [4]:
for ext in ("pgs", "shex", "ttl"):
    print(f"--- references.{ext}")
    print((SCHEMAS / "equivalence" / f"references.{ext}").read_text())

--- references.pgs
CREATE NODE TYPE ( PersonType : Person {
    name: STRING,
    OPTIONAL age: INTEGER CHECK > 17
}) ;
CREATE NODE TYPE ( OrgType : Org {
    label: STRING
}) ;
CREATE NODE TYPE ( CityType : City {
    name: STRING
}) ;
CREATE EDGE TYPE (:@PersonType) -[WorksForType : worksFor]-> (:@OrgType) ;
CREATE EDGE TYPE (:@OrgType) -[LocatedInType : locatedIn]-> (:@CityType)

--- references.shex
# references.pgs translated to ShEx over its RDF encoding, keeping PG-Schema's defaults:
# closed records become CLOSED shapes, and keys without OPTIONAL stay required.
PREFIX t: <http://rudof.example/pg/type/>
PREFIX k: <http://rudof.example/pg/key/>
PREFIX l: <http://rudof.example/pg/label/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

t:PersonType CLOSED {
  a [ l:Person ] ;
  k:name xsd:string ;
  k:age xsd:integer MinExclusive 17 ? ;
  <http://rudof.example/pg/edge/WorksForType/worksFor/conforming-source> @t:OrgType *
}
t:OrgType CLOSED {
  a [ l:Org ] ;
  k:label xsd:string ;
 

In [5]:
def generate_three(name, entities=30, seed=7):
    """Generate from <name>.pgs, .shex and .ttl with the same seed; return the RDF paths."""
    base = SCHEMAS / "equivalence" / name
    pg = config(entities, seed, path=OUT / f"{name}.pg", format=OutputFormat.Pg, shapemap=OUT / f"{name}.map")
    pg.set_rdf_output(OUT / f"{name}-pg.ttl", OUT / f"{name}-pg.shapemap")
    DataGenerator(pg).run_with_format(base.with_suffix(".pgs"), SchemaFormat.PgSchema)
    DataGenerator(config(entities, seed, path=OUT / f"{name}-shex.ttl", shapemap=OUT / f"{name}-shex.shapemap")).run_with_format(base.with_suffix(".shex"), SchemaFormat.ShEx)
    DataGenerator(config(entities, seed, path=OUT / f"{name}-shacl.ttl")).run_with_format(base.with_suffix(".ttl"), SchemaFormat.Shacl)
    return OUT / f"{name}-pg.ttl", OUT / f"{name}-shex.ttl", OUT / f"{name}-shacl.ttl"


pg_rdf, shex_rdf, shacl_rdf = generate_three("references")
a, b, c = ntriples(pg_rdf), ntriples(shex_rdf), ntriples(shacl_rdf)
print(f"triples: PG-Schema {len(a)}, ShEx {len(b)}, SHACL {len(c)}")
print("identical:", a == b == c)

triples: PG-Schema 75, ShEx 75, SHACL 75


identical: True


The three graphs are identical, triple for triple. Each is also valid against all three
schemas — this is how the paper establishes that constructs outside the common fragment of
the three languages (here, references between node types) are shared: they generate the
same data.

In [6]:
schema = lambda ext: (SCHEMAS / "equivalence" / f"references.{ext}").read_text(encoding="utf-8")

with Rudof() as rudof:
    rudof.read_shex(schema("shex"), ShExFormat.ShExC)
    rudof.read_data(pg_rdf.read_text(encoding="utf-8"), RDFFormat.Turtle)
    rudof.read_shapemap((OUT / "references-pg.shapemap").read_text(encoding="utf-8"), ShapeMapFormat.Compact)
    print("ShEx  :", rudof.validate_shex().conforms)

with Rudof() as rudof:
    rudof.read_shacl(schema("ttl"), ShaclFormat.Turtle)
    rudof.read_data(pg_rdf.read_text(encoding="utf-8"), RDFFormat.Turtle)
    print("SHACL :", rudof.validate_shacl(ShaclValidationMode.Native).conforms)

with Rudof() as rudof:
    rudof.read_data((OUT / "references.pg").read_text(encoding="utf-8"), RDFFormat.Pg)
    rudof.read_pgschema(schema("pgs"), PgSchemaFormat.PgSchemaC)
    rudof.read_typemap((OUT / "references.map").read_text(encoding="utf-8"))
    print("PG-Schema:", rudof.validate_pgschema().conforms)

ShEx  : True
SHACL : True
PG-Schema: True


## 3. One graph, two data models

From the PG-Schema, the generator writes the graph as a property graph (`.pg`), with a
**type map** that assigns every node and edge its type, and the same graph as RDF.

In [7]:
print("\n".join((OUT / "references.pg").read_text().splitlines()[:6]))
print("...")
print("\n".join((OUT / "references.map").read_text().splitlines()[:4]))

(CityType_1 {City} [ name: "Alpha349" ])
(CityType_10 {City} [ name: "Alpha944" ])
(CityType_2 {City} [ name: "Delta342" ])
(CityType_3 {City} [ name: "Epsilon847" ])
(CityType_4 {City} [ name: "Delta437" ])
(CityType_5 {City} [ name: "Alpha748" ])
...
CityType_1: CityType,
CityType_10: CityType,
CityType_2: CityType,
CityType_3: CityType,


## 4. A published schema: LDBC SNB

The LDBC Social Network Benchmark schema as a PG-Schema (11 node types; the 19 edge types
without edge properties are generated), with ShEx and SHACL translations. We generate it
from the three languages and measure its **coherence** (Duan et al., SIGMOD 2011): for each
type, the share of the type's properties its instances have, weighted by type size.

In [8]:
def coherence(triples):
    rdf_type = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
    types, props = defaultdict(set), defaultdict(set)
    for t in triples:
        s, p, o = t.split(" ", 2)
        if p == rdf_type:
            types[o.rstrip(" .")].add(s)
        else:
            props[s].add(p)
    total, weighted = 0, 0.0
    for instances in types.values():
        p_t = set().union(*(props[s] for s in instances))
        if not p_t:
            continue
        cv = sum(len(props[s]) for s in instances) / (len(p_t) * len(instances))
        weight = len(p_t) + len(instances)
        total, weighted = total + weight, weighted + weight * cv
    return weighted / total


snb = [ntriples(p) for p in generate_three("ldbc-snb", entities=1100)]
print(f"triples: {[len(t) for t in snb]}, identical: {snb[0] == snb[1] == snb[2]}")
print(f"coherence: {coherence(snb[0]):.3f}")

triples: [6700, 6700, 6700], identical: True
coherence: 0.855


## 5. Controlled errors and their report

On request, the generator breaks a share of the entities — each with one violation of one
constraint — and writes a JSON Lines **report**: every node (and, for a property graph,
edge) that a correct validator must flag. Failures propagate through references, and the
languages place them differently: in a property graph a broken organisation fails the
**edges** that require it to conform; in RDF it fails the **persons** who work for it, and
so on transitively. The generator writes a report for each reading.

In [9]:
def broken(name, rate=0.2, entities=300, seed=11, closed=True, keys_required=True):
    c = config(entities, seed, path=OUT / f"{name}-broken.pg", format=OutputFormat.Pg, shapemap=OUT / f"{name}-broken.map")
    c.set_rdf_output(OUT / f"{name}-broken.ttl", OUT / f"{name}-broken.shapemap")
    c.set_violations(rate, OUT / f"{name}-labels.jsonl")
    c.set_rdf_labels(OUT / f"{name}-rdf-labels.jsonl", closed=closed, keys_required=keys_required)
    DataGenerator(c).run_with_format(SCHEMAS / "equivalence" / f"{name}.pgs", SchemaFormat.PgSchema)
    read = lambda p: [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l]
    return read(OUT / f"{name}-labels.jsonl"), read(OUT / f"{name}-rdf-labels.jsonl")


pg_labels, rdf_labels = broken("references")
print("property graph report:", Counter(l["kind"] for l in pg_labels))
print("RDF report           :", Counter(l["kind"] for l in rdf_labels))
print("operators            :", Counter(l.get("operator") for l in pg_labels if l["kind"] == "direct"))
print(json.dumps(rdf_labels[0], indent=1))

property graph report: Counter({'direct': 58, 'induced': 30})
RDF report           : Counter({'direct': 58, 'induced': 22})
operators            : Counter({'drop-required': 32, 'closed-extra': 16, 'dangling-reference': 4, 'exceed-max': 3, 'wrong-datatype': 2, 'out-of-range': 1})
{
 "node": "http://rudof.example/pg/type/CityType-11",
 "shape": "http://rudof.example/pg/type/CityType",
 "kind": "direct",
 "property": "http://rudof.example/violations#undeclaredProperty",
 "operator": "closed-extra"
}


Each validator must report exactly the failing nodes of its report:

In [10]:
failing = lambda labels: {l["node"] for l in labels if l["kind"] != "trap"}

with Rudof() as rudof:
    rudof.read_data((OUT / "references-broken.pg").read_text(encoding="utf-8"), RDFFormat.Pg)
    rudof.read_pgschema(schema("pgs"), PgSchemaFormat.PgSchemaC)
    rudof.read_typemap((OUT / "references-broken.map").read_text(encoding="utf-8"))
    pg_reported = {v.node_id for v in rudof.validate_pgschema().violations}
print("PG-Schema validator == property graph report:", pg_reported == failing(pg_labels))

with Rudof() as rudof:
    rudof.read_shacl(schema("ttl"), ShaclFormat.Turtle)
    rudof.read_data((OUT / "references-broken.ttl").read_text(encoding="utf-8"), RDFFormat.Turtle)
    report = rudof.validate_shacl(ShaclValidationMode.Native)
    shacl_reported = {str(e.focus_node).strip("<>") for e in report}
print("SHACL validator == RDF report:", shacl_reported == failing(rdf_labels))

PG-Schema validator == property graph report: True
SHACL validator == RDF report: True


## 6. When the languages' defaults differ

PG-Schema records are closed and keys required unless declared `OPTIONAL`; a SHACL modeller
would usually leave shapes open and properties optional. Read through such a translation,
an undeclared key or a dropped key is harmless — the report marks it as a **trap**, a change
a correct validator must *not* flag.

In [11]:
_, idiomatic = broken("references", closed=False, keys_required=False)
print("RDF report, idiomatic SHACL translation:", Counter(l["kind"] for l in idiomatic))

with Rudof() as rudof:
    rudof.read_shacl((SCHEMAS / "cross-language" / "references-idiomatic.ttl").read_text(encoding="utf-8"), ShaclFormat.Turtle)
    rudof.read_data((OUT / "references-broken.ttl").read_text(encoding="utf-8"), RDFFormat.Turtle)
    reported = {str(e.focus_node).strip("<>") for e in rudof.validate_shacl(ShaclValidationMode.Native)}
print("SHACL validator == idiomatic report:", reported == failing(idiomatic))

RDF report, idiomatic SHACL translation: Counter({'trap': 52, 'direct': 6, 'induced': 2})


SHACL validator == idiomatic report:

 True


## 7. Configuration files

Everything above can also be written as a configuration (format v2):

In [12]:
toml = f'''
version = 2
[run]
seed = 7
scale = {{ entities = 200 }}
[violations]
rate = 0.1
operators = ["drop-required", "dangling-reference"]
labels = "{(OUT / 'v2-labels.jsonl').as_posix()}"
[output]
format = "pg"
path = "{(OUT / 'v2.pg').as_posix()}"
shapemap = "{(OUT / 'v2.map').as_posix()}"
stats = false
'''
DataGenerator(GeneratorConfig.from_toml_str(toml)).run_with_format(SCHEMAS / "violations" / "persons.pgs", SchemaFormat.PgSchema)
print(Counter(json.loads(l)["operator"] for l in (OUT / "v2-labels.jsonl").read_text().splitlines() if l and json.loads(l)["kind"] == "direct"))

Counter({'drop-required': 17, 'dangling-reference': 4})
